# Q1. 423. Truncated BPTT




Truncated Backpropagation Through Time (BPTT) is a practical way to train sequence models (like vanilla RNNs) on long sequences without backpropagating through the entire history. In this problem, you’ll implement one training step that computes gradients using a fixed truncation window 
k
k, instead of the full sequence length.

Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
⌄
⌄
import numpy as np

def truncated_bptt_step(xs, ys, Wxh, Whh, Why, bh, by, h0, k, lr):
    """
    Performs one training step for a simple RNN using truncated BPTT.

    RNN equations (for t = 0..T-1):
        h_t = tanh(Wxh x_t + Whh h_{t-1} + b_h)
        o_t = Why h_t + b_y
        p_t = softmax(o_t)

    Loss (average over time):
        L = -(1/T) * sum_t log(p_t[y_t])

    Truncation:
        Only backpropagate gradients through the last k time steps:
        i.e., when computing gradients at time t, stop at time max(0, t-k+1).

    Args:
        xs (np.ndarray): Inputs, shape (T, input_dim).
        ys (np.ndarray): Target class indices, shape (T,).
        Wxh (np.ndarray): (hidden_dim, input_dim)
        Whh (np.ndarray): (hidden_dim, hidden_dim)
        Why (np.ndarray): (output_dim, hidden_dim)
        bh (np.ndarray): (hidden_dim,)
        by (np.ndarray): (output_dim,)
        h0 (np.ndarray): Initial hidden state (hidden_dim,)
        k (int): Truncation window size.
        lr (float): Learning rate.

    Returns:
        tuple:
            new_Wxh, new_Whh, new_Why, new_bh, new_by (all np.ndarray),
            loss (float),
            hT (np.ndarray): final hidden state after processing the full sequence.
    """
    # Your solution below
Rules:

Use the equations above and compute the average cross-entropy loss:
L
=
−
1
T
∑
t
=
0
T
−
1
log
⁡
p
t
[
y
t
]
L=− 
T
1
​
  
t=0
∑
T−1
​
 logp 
t
​
 [y 
t
​
 ]
Implement a numerically stable softmax (subtract max logit before exponentiating).
Implement truncated BPTT: only propagate gradients back at most k steps in time.
Update parameters with one step of vanilla SGD: param -= lr * grad.
Use only NumPy + Python built-ins; return parameters as NumPy arrays.
Example
python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
⌄
⌄
⌄
⌄
xs = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
ys = np.array([0, 1, 0])

Wxh = np.array([[0.1, -0.2],
                [0.0,  0.3]])
Whh = np.array([[0.05, 0.0],
                [0.0,  0.05]])
Why = np.array([[0.2, -0.1],
                [-0.2, 0.1]])
bh = np.array([0.0, 0.0])
by = np.array([0.0, 0.0])
h0 = np.array([0.0, 0.0])

k = 2
lr = 0.1

new_Wxh, new_Whh, new_Why, new_bh, new_by, loss, hT = truncated_bptt_step(
    xs, ys, Wxh, Whh, Why, bh, by, h0, k, lr
)
Output:

python
Copy
1
2
loss  # a single float value (e.g., 0.6...)
hT    # final hidden state as a NumPy array (length hidden_dim)


In [ ]:
import numpy as np


def truncated_bptt_step(xs, ys, Wxh, Whh, Why, bh, by, h0, k, lr):
    # Helper to avoid mutating inputs
    Wxh = Wxh.astype(float).copy()
    Whh = Whh.astype(float).copy()
    Why = Why.astype(float).copy()
    bh = bh.astype(float).copy()
    by = by.astype(float).copy()

    T = xs.shape[0]
    hidden_dim = h0.shape[0]
    output_dim = by.shape[0]

    # Forward pass storage
    hs = np.zeros((T + 1, hidden_dim))
    hs[0] = h0
    os = np.zeros((T, output_dim))
    ps = np.zeros((T, output_dim))

    # Forward pass
    for t in range(T):
        x_t = xs[t]
        h_prev = hs[t]
        hs[t + 1] = np.tanh(Wxh @ x_t + Whh @ h_prev + bh)
        os[t] = Why @ hs[t + 1] + by
        o_t = os[t] - np.max(os[t])
        exp_o = np.exp(o_t)
        ps[t] = exp_o / np.sum(exp_o)

    # Average cross-entropy loss
    log_probs = -np.log(ps[np.arange(T), ys])
    loss = np.mean(log_probs)

    # Gradients
    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)
    dbh = np.zeros_like(bh)
    dby = np.zeros_like(by)

    # Backward pass with truncation
    for t in range(T - 1, -1, -1):
        dp = ps[t].copy()
        dp[ys[t]] -= 1.0
        dp /= T

        dWhy += np.outer(dp, hs[t + 1])
        dby += dp

        dh = Why.T @ dp
        t_start = max(0, t - k + 1)

        for tb in range(t, t_start - 1, -1):
            h_tb = hs[tb + 1]
            h_prev = hs[tb]
            dtanh = (1.0 - h_tb ** 2) * dh

            dbh += dtanh
            dWxh += np.outer(dtanh, xs[tb])
            dWhh += np.outer(dtanh, h_prev)

            dh = Whh.T @ dtanh

    # SGD update
    Wxh -= lr * dWxh
    Whh -= lr * dWhh
    Why -= lr * dWhy
    bh -= lr * dbh
    by -= lr * dby

    return (
        Wxh,
        Whh,
        Why,
        bh,
        by,
        float(loss),
        hs[-1],
    )

result = truncated_bptt_step(xs, ys, Wxh, Whh, Why, bh, by, h0, k, lr)
result